# Predicting deprivation

## Idea

We are going to try to use satellite data to predict deprivation and show that, indeed, satellites are better than old data, and embeddings are better than manual features.

### Outline

1. Data prep
    - Landsat: obtain cloud-free images/composites for London extent for 2019 and 2025
    - Embeddings: download our [data product](https://data.imago.ac.uk/datasets/google-satellite-embedding-v1-london-lsoas-2020-2024)
    - Geometries: use those from the embeddings files
1. Setup
    - Train/test split
    - Performance ($R^2$, $RMSE$)
1. Model fitting (2019)
    - m1/ `IMD19 ~ Manual features`
    - m2/ `IMD19 ~ Embeddings`
1. _Who is better at predicting contemporaneous IMD?_ --> Model comparison (m1 Vs. m2), winner?
1. _What about _future_ IMD?_
    - Build predictions for 2025 with m1 ($\hat{m1}$) and m2 ($\hat{m2}$)
    - Compare $\hat{m1}$, $\hat{m2}$, _and_ $IMD_{19}$

## Data

- LSOAs for London (2020 def)
- IMD
    - 2019
    - 2025
- Landsat to build features
    - 2019
    - 2025
- LSOA Embeddings
    - 2019
    - 2024

Resources:

- IMD [2019](https://www.gov.uk/government/statistics/english-indices-of-deprivation-2019) and [2025](https://deprivation.communities.gov.uk/download-all)
- [GeoVisualisation lab](https://gdsl-ul.github.io/wma/labs/w07_pixelsToPatterns.html)
- [London embeddings](https://data.imago.ac.uk/datasets/google-satellite-embedding-v1-london-lsoas-2020-2024)

In [6]:
import pandas
import geopandas
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import GroupKFold

# Load data
imd = pandas.read_csv('imd.csv', index_col='LSOA21CD')
emb = geopandas.read_file('uk_lsoa_london_embeds_2020.geojson').set_index('LSOA21CD')

emb_cols = [c for c in emb.columns if c.endswith('_mean')]
db = imd[['imd19_score', 'led19_score', 'grid_id']].join(emb[emb_cols])

gkf = GroupKFold(n_splits=5)

for target in ['imd19_score', 'led19_score']:
    sub = db.dropna(subset=[target] + emb_cols)
    X = sub[emb_cols].values
    y = sub[target].values
    groups = sub['grid_id'].values

    fold_r2, fold_rmse = [], []
    for tr_idx, te_idx in gkf.split(X, y, groups):
        model = HistGradientBoostingRegressor(random_state=42)
        model.fit(X[tr_idx], y[tr_idx])
        y_pred = model.predict(X[te_idx])
        fold_r2.append(r2_score(y[te_idx], y_pred))
        fold_rmse.append(root_mean_squared_error(y[te_idx], y_pred))

    print(
        f"{target}  —  "
        f"R²: {np.mean(fold_r2):.3f} ± {np.std(fold_r2):.3f}  |  "
        f"RMSE: {np.mean(fold_rmse):.3f} ± {np.std(fold_rmse):.3f}"
    )


imd19_score  —  R²: 0.104 ± 0.077  |  RMSE: 10.145 ± 0.381
led19_score  —  R²: 0.304 ± 0.081  |  RMSE: 8.454 ± 0.989


In [7]:
# 2025 IMD — embeddings from 2024, targets are ranks
emb24 = geopandas.read_file('uk_lsoa_london_embeds_2024.geojson').set_index('LSOA21CD')
emb_cols24 = [c for c in emb24.columns if c.endswith('_mean')]
db24 = imd[['imd25_rank', 'led25_rank', 'grid_id']].join(emb24[emb_cols24])

for target in ['imd25_rank', 'led25_rank']:
    sub = db24.dropna(subset=[target] + emb_cols24)
    X = sub[emb_cols24].values
    y = sub[target].values
    groups = sub['grid_id'].values

    fold_r2, fold_rmse = [], []
    for tr_idx, te_idx in gkf.split(X, y, groups):
        model = HistGradientBoostingRegressor(random_state=42)
        model.fit(X[tr_idx], y[tr_idx])
        y_pred = model.predict(X[te_idx])
        fold_r2.append(r2_score(y[te_idx], y_pred))
        fold_rmse.append(root_mean_squared_error(y[te_idx], y_pred))

    print(
        f"{target}  —  "
        f"R²: {np.mean(fold_r2):.3f} ± {np.std(fold_r2):.3f}  |  "
        f"RMSE: {np.mean(fold_rmse):.3f} ± {np.std(fold_rmse):.3f}"
    )


imd25_rank  —  R²: 0.093 ± 0.173  |  RMSE: 7896.497 ± 306.573
led25_rank  —  R²: 0.359 ± 0.085  |  RMSE: 4773.625 ± 494.099


In [12]:
from scipy.stats import spearmanr

# Fit final models on all 2019 data (no CV — we're forecasting, not evaluating 2019)
db19_full = imd[['imd19_score', 'led19_score']].join(emb[emb_cols]).dropna()
db25_full = imd[['imd25_rank', 'led25_rank']].join(emb24[emb_cols24]).dropna()

for target19, target25 in [('imd19_score', 'imd25_rank'), ('led19_score', 'led25_rank')]:
    model = HistGradientBoostingRegressor(random_state=42)
    model.fit(db19_full[emb_cols].values, db19_full[target19].values)

    # Predict 2025 scores; invert to ranks so high score → rank 1 (most deprived),
    # matching imd25_rank convention where 1 = most deprived
    common = db25_full.index.intersection(db19_full.index)
    sub25 = db25_full.loc[common]
    pred_ranks = (
        pandas.Series(model.predict(sub25[emb_cols24].values), index=sub25.index)
        .rank(ascending=False)
    )
    actual_ranks = sub25[target25].rank()

    rho, pval = spearmanr(actual_ranks, pred_ranks)
    r2 = r2_score(actual_ranks, pred_ranks)
    print(
        f"{target19} → {target25}  —  "
        f"Spearman ρ: {rho:.3f} (p={pval:.2e})  |  R²: {r2:.3f}"
    )


imd19_score → imd25_rank  —  Spearman ρ: 0.564 (p=0.00e+00)  |  R²: 0.128
led19_score → led25_rank  —  Spearman ρ: 0.637 (p=0.00e+00)  |  R²: 0.275
